# 01 — Data preparation (C4)

Beginner path: install packages → get C4 source into the runtime → load the **real** ASSISTments corrected CSV → quality report → learner splits → fit response model on **train only**.

**Expected outputs** (under your output/Drive folder):
- `artifacts/quality_report.json`
- `artifacts/real_data_provenance.json`
- `artifacts/split_meta.json`
- `artifacts/response_model.joblib`
- `artifacts/skill_catalogue.json`
- feature pickle files

This notebook does **not** train RL policies.


## Step 0 — Configuration

Edit these values before running other cells.


In [ ]:
# === USER CONFIG ===
RUN_IN_COLAB = True  # set False for local Cursor/Jupyter
DRIVE_SUBDIR = "AdaptiveLearnSE/c4_runs"  # under Google Drive MyDrive
ALLOW_GDOWN = False  # True attempts Drive file id from the official ASSISTments page
# Path to CSV after manual upload, or None to search default locations
CSV_PATH = None  # e.g. "/content/skill_builder_data_corrected.csv"
TOP_K_SKILLS = 20
SEED = 42
SOURCE_MODE = "zip"  # "zip" | "github" | "local"
GITHUB_URL = "https://github.com/thumashi2001/J26-SE-317.git"
GITHUB_BRANCH = "feature/c4-adaptive-learning"
GITHUB_COMMIT = None  # set to a verified SHA when using github mode


## Step 1 — Install training dependencies (run before project imports)


In [ ]:
import sys
print(sys.executable, sys.version)


## Step 2 — Obtain C4 source (ZIP recommended for unpushed work)


In [ ]:
from pathlib import Path
import sys
import subprocess

C4_ROOT = None
if SOURCE_MODE == "local":
    C4_ROOT = Path("..").resolve()
elif RUN_IN_COLAB and SOURCE_MODE == "zip":
    from google.colab import files
    import zipfile
    print("Upload c4-source.zip (repo or ai-services/c4)...")
    up = files.upload()
    name = next(iter(up))
    dest = Path("/content/J26-SE-317")
    dest.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(name) as zf:
        zf.extractall(dest)
    candidates = [dest / "ai-services" / "c4", dest]
    for child in dest.iterdir():
        if child.is_dir():
            candidates.append(child / "ai-services" / "c4")
            candidates.append(child)
    for c in candidates:
        if (c / "training").exists():
            C4_ROOT = c
            break
    if C4_ROOT is None:
        raise FileNotFoundError("Could not locate ai-services/c4 after ZIP extract")
elif RUN_IN_COLAB and SOURCE_MODE == "github":
    import shutil
    dest = Path("/content/J26-SE-317")
    if dest.exists():
        shutil.rmtree(dest)
    cmd = ["git", "clone", "--branch", GITHUB_BRANCH, "--single-branch", GITHUB_URL, str(dest)]
    subprocess.check_call(cmd)
    if GITHUB_COMMIT:
        subprocess.check_call(["git", "checkout", GITHUB_COMMIT], cwd=str(dest))
    C4_ROOT = dest / "ai-services" / "c4"
else:
    C4_ROOT = Path("..").resolve()

assert (C4_ROOT / "training").exists(), C4_ROOT
sys.path.insert(0, str(C4_ROOT))
print("C4_ROOT =", C4_ROOT)

req = C4_ROOT / "requirements-training.txt"
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-r", str(req)])
print("Dependencies installed from", req)


## Step 3 — Output directories (Drive if available)


In [ ]:
from training.paths import resolve_output_root, stage_dirs
from training.colab_bootstrap import mount_drive

if RUN_IN_COLAB:
    try:
        mount_drive()
    except Exception as e:
        print("Drive mount skipped/failed:", e)

OUT = resolve_output_root(DRIVE_SUBDIR if RUN_IN_COLAB else None, prefer_drive=RUN_IN_COLAB)
DIRS = stage_dirs(OUT)
ARTIFACTS = DIRS["artifacts"]
print("Output root:", OUT)
print("Artifacts:", ARTIFACTS)


## Step 4 — Acquire ASSISTments corrected CSV

Official page: https://sites.google.com/site/assistmentsdata/home/2009-2010-assistment-data/skill-builder-data-2009-2010

Corrected Drive link from that page: https://drive.google.com/file/d/1NNXHFRxcArrU0ZJSb9BIL56vmUt5FhlE/view?usp=sharing

If you need to upload manually in Colab, run the next cell.


In [ ]:
# Optional manual CSV upload (Colab)
if RUN_IN_COLAB and CSV_PATH is None:
    from google.colab import files
    print("If the CSV is not already on disk, upload skill_builder_data_corrected.csv now (or cancel to skip).")
    try:
        up = files.upload()
        if up:
            CSV_PATH = list(up.keys())[0]
            print("Using uploaded", CSV_PATH)
    except Exception as e:
        print("Upload skipped:", e)


## Step 5 — Run data preparation


In [ ]:
from training.pipeline import run_data_preparation
from training.config import SplitConfig

result = run_data_preparation(
    csv_path=CSV_PATH,
    output_dir=ARTIFACTS,
    split_config=SplitConfig(seed=SEED),
    allow_download=ALLOW_GDOWN,
    top_k_skills=TOP_K_SKILLS,
    seed=SEED,
)
print("Done. Keys:", sorted(result.keys()))
print("Learners:", result["split_meta"]["n_learners"])
print("Response val metrics:", result["response_metrics"]["validation"])
print("IMPORTANT: test metrics are held out — do not retune on them.")
print(result["response_metrics"]["test_held_out"])


## Step 6 — Quick peek at the quality report


In [ ]:
import json
qr = json.loads((ARTIFACTS / "quality_report.json").read_text())
print("columns:", len(qr["available_columns"]))
print("learners:", qr["n_learners"], "interactions:", qr["n_normalised_interactions"])
print("multi-skill fraction:", qr["multi_skill"]["fraction"])
print("domain_warning:", qr["domain_warning"])
